# 01.05 · Calculus & gradient descent

**Chapter:** Python & Math for AI · **Milestone:** M1

## Goals
- [ ] Derivative as "slope", numerically and analytically
- [ ] Chain rule — the entire idea behind backpropagation
- [ ] Gradients of multi-variable functions; gradient descent in 1-D and 2-D
- [ ] Feel the effect of the learning rate

## Resources
- [3Blue1Brown – Essence of calculus, ch. 1-4](https://www.3blue1brown.com/topics/calculus) — free
- [3Blue1Brown – Neural networks, ch. 2-4 (gradient descent, backprop)](https://www.3blue1brown.com/topics/neural-networks) — free

Training a model = adjusting millions of numbers to make a loss smaller. The gradient tells you which direction makes it smaller.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Derivatives
$f'(x) = \lim_{h \to 0} \frac{f(x+h) - f(x)}{h}$: how much $f$ changes per unit change of $x$.
Numerical derivatives are slow and approximate, but perfect for **checking** analytic ones.

In [ ]:
def f(x):
    return x**3 - 2 * x + 1

def f_prime(x):  # analytic: d/dx (x^3 - 2x + 1)
    return 3 * x**2 - 2

def numerical_derivative(fn, x, h=1e-5):
    return (fn(x + h) - fn(x - h)) / (2 * h)  # central difference: more accurate

for x in [-1.0, 0.0, 2.0]:
    print(x, f_prime(x), round(numerical_derivative(f, x), 6))

## 2. The chain rule
If $y = f(g(x))$ then $\frac{dy}{dx} = f'(g(x)) \cdot g'(x)$. A neural network is a long chain of functions;
backprop is just applying the chain rule from the loss backwards (you'll build it in 03.01).

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# y = sigmoid(w * x + b) ; dy/dw = sigmoid'(wx+b) * x  where sigmoid'(z) = s(z)(1-s(z))
w, b, x = 0.5, -1.0, 3.0
z = w * x + b
dy_dw = sigmoid(z) * (1 - sigmoid(z)) * x
print("chain rule:", round(dy_dw, 6), "| numerical:", round(numerical_derivative(lambda w_: sigmoid(w_ * x + b), w), 6))

## 3. Gradient descent in 1-D
Repeat: $x \leftarrow x - \eta \, f'(x)$. The **learning rate** $\eta$ is the most important hyperparameter you'll tune.

In [ ]:
def descend(x0, lr, steps=30):
    xs = [x0]
    for _ in range(steps):
        xs.append(xs[-1] - lr * f_prime(xs[-1]))
        if abs(xs[-1]) > 10:  # diverged: stop before floats overflow
            break
    return np.array(xs)

grid = np.linspace(-2, 2, 200)
fig, axes = plt.subplots(1, 3, figsize=(13, 3))
for ax, lr in zip(axes, [0.01, 0.1, 0.45]):
    xs = descend(1.8, lr)
    ax.plot(grid, f(grid), "k-", alpha=0.4)
    ax.plot(xs, f(xs), "ro-", ms=3)
    ax.set_xlim(-2.2, 2.2); ax.set_ylim(-1, 5); ax.set_title(f"lr={lr}: ends at x={xs[-1]:.3f}")
plt.tight_layout(); plt.show()
print("true local minimum at x =", round(np.sqrt(2 / 3), 3))

Small lr: slow. Good lr: converges. Too big: overshoots and diverges (x flies off the plot).

## 4. Gradients in 2-D
For $f(x, y)$, the **gradient** $\nabla f = (\partial f/\partial x, \partial f/\partial y)$ points uphill.
Below: an elongated bowl, like real loss surfaces where some directions are much steeper than others
(which is why adaptive optimizers like Adam exist, chapter 03).

In [ ]:
def loss2d(p):
    x, y = p
    return 0.5 * x**2 + 5 * y**2

def grad2d(p):
    x, y = p
    return np.array([x, 10 * y])

X, Y = np.meshgrid(np.linspace(-6, 6, 200), np.linspace(-3, 3, 200))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, lr in zip(axes, [0.05, 0.19]):
    p = np.array([-5.5, 2.5]); path = [p]
    for _ in range(40):
        p = p - lr * grad2d(p); path.append(p)
    path = np.array(path)
    ax.contour(X, Y, loss2d((X, Y)), levels=20, alpha=0.5)
    ax.plot(path[:, 0], path[:, 1], "ro-", ms=3)
    ax.set_title(f"lr={lr}, final loss={loss2d(path[-1]):.4f}")
plt.tight_layout(); plt.show()

With the bigger lr the steep (y) direction zig-zags while the shallow (x) direction is still slow — the classic
ill-conditioning problem. Momentum and Adam fix this.

## Exercises
1. Generate data `y = 3x + 2 + noise`. Fit `a, b` with hand-written gradient descent on MSE loss
   (derive $\partial L/\partial a$ and $\partial L/\partial b$ yourself). Plot loss vs step.
2. Check your gradients with `numerical_derivative` before training (always do this when you hand-derive gradients).
3. Add **momentum** (`v = β v + grad; p -= lr * v`) to the 2-D example. Does it fix the zig-zag?

In [ ]:
rng = np.random.default_rng(0)
xs = rng.uniform(-3, 3, 200)
ys = 3 * xs + 2 + rng.normal(0, 0.5, 200)
# your code here